# ML-KEM-768 Driver Notebook

Interactive notebook wrapper for the unified RTL accelerator (`ml_kem_top`).

## Notes
- Place `ml_kem.bit` and `ml_kem.hwh` in the same folder and keep the same basename.
- Default IP instance name is `ml_kem_top_0`.

## AXI-Lite Register Map

| Offset | Name | Access | Meaning |
|---|---|---|---|
| `0x00` | `CTRL` | W | `[0]=start`, `[2:1]=op_sel` (`00` KeyGen, `01` Encaps, `10` Decaps) |
| `0x04` | `STATUS` | R | `[0]=done`, `[1]=idle`, `[2]=error` |
| `0x08` | `CYCLES` | R | 32-bit cycle counter |
| `0x10..0x2C` | `SEED_D[0..7]` | W | 8 x 32-bit words (little-endian) |
| `0x30..0x4C` | `SEED_Z[0..7]` | W | 8 x 32-bit words (little-endian) |
| `0x50` | `PK_ADDR` | W | DDR physical address (pk, 1184 bytes) |
| `0x54` | `SK_ADDR` | W | DDR physical address (sk, 2400 bytes) |
| `0x58` | `CT_ADDR` | W | DDR physical address (ct, 1088 bytes) |
| `0x5C` | `SS_ADDR` | W | DDR physical address (ss, 32 bytes) |
| `0x60` | `M_ADDR`  | W | DDR physical address (m, 32 bytes) |


In [ ]:
import struct
import time

import numpy as np
from pynq import Overlay, allocate

REG_CTRL         = 0x00
REG_STATUS       = 0x04
REG_CYCLES       = 0x08
REG_SEED_D_BASE  = 0x10
REG_SEED_Z_BASE  = 0x30
REG_PK_ADDR      = 0x50
REG_SK_ADDR      = 0x54
REG_CT_ADDR      = 0x58
REG_SS_ADDR      = 0x5C
REG_M_ADDR       = 0x60

CTRL_START_KEYGEN = 0b001
CTRL_START_ENCAPS = 0b011
CTRL_START_DECAPS = 0b101

STATUS_DONE  = 0x1
STATUS_IDLE  = 0x2
STATUS_ERROR = 0x4

PK_SIZE = 1184
SK_SIZE = 2400
CT_SIZE = 1088
SS_SIZE = 32
M_SIZE  = 32

PL_CLK_HZ = 100_000_000


class MLKemError(RuntimeError):
    """Raised when the accelerator reports an error or times out."""


In [ ]:
class MLKem768:
    """Driver for unified ML-KEM-768 RTL accelerator."""

    DEFAULT_TIMEOUTS_S = {
        "keygen": 2.0,
        "encaps": 2.0,
        "decaps": 3.0,
    }

    def __init__(self, bitfile, ip_name="ml_kem_top_0"):
        self.overlay = Overlay(bitfile)
        if not hasattr(self.overlay, ip_name):
            raise MLKemError(
                f"IP '{ip_name}' not found. Available: {list(self.overlay.ip_dict.keys())}"
            )
        self.ip = getattr(self.overlay, ip_name)

        self.pk = allocate((PK_SIZE,), dtype=np.uint8)
        self.sk = allocate((SK_SIZE,), dtype=np.uint8)
        self.ct = allocate((CT_SIZE,), dtype=np.uint8)
        self.ss = allocate((SS_SIZE,), dtype=np.uint8)
        self.m  = allocate((M_SIZE,),  dtype=np.uint8)

        self.ip.write(REG_PK_ADDR, self.pk.physical_address)
        self.ip.write(REG_SK_ADDR, self.sk.physical_address)
        self.ip.write(REG_CT_ADDR, self.ct.physical_address)
        self.ip.write(REG_SS_ADDR, self.ss.physical_address)
        self.ip.write(REG_M_ADDR,  self.m.physical_address)

    def _write_seed(self, base_offset, seed_bytes):
        if len(seed_bytes) != 32:
            raise ValueError(f"Seed must be 32 bytes, got {len(seed_bytes)}")
        for i in range(8):
            word = struct.unpack_from("<I", seed_bytes, i * 4)[0]
            self.ip.write(base_offset + i * 4, word)

    def _wait_done(self, timeout_s):
        deadline = time.monotonic() + timeout_s
        while True:
            status = self.ip.read(REG_STATUS)
            if status & STATUS_ERROR:
                raise MLKemError(f"IP signaled error, STATUS=0x{status:x}")
            if status & STATUS_DONE:
                return self.ip.read(REG_CYCLES)
            if time.monotonic() > deadline:
                raise MLKemError(
                    f"Operation timeout after {timeout_s}s, STATUS=0x{status:x}"
                )

    def _ensure_idle(self):
        status = self.ip.read(REG_STATUS)
        if not (status & STATUS_IDLE):
            raise MLKemError(f"IP not idle before new op, STATUS=0x{status:x}")

    def keygen(self, seed_d, seed_z, timeout_s=None):
        self._ensure_idle()
        self._write_seed(REG_SEED_D_BASE, seed_d)
        self._write_seed(REG_SEED_Z_BASE, seed_z)
        self.ip.write(REG_CTRL, CTRL_START_KEYGEN)
        cycles = self._wait_done(
            timeout_s if timeout_s is not None else self.DEFAULT_TIMEOUTS_S["keygen"]
        )
        self.pk.invalidate()
        self.sk.invalidate()
        return bytes(self.pk), bytes(self.sk), cycles

    def encaps(self, pk_bytes, m_bytes, timeout_s=None):
        if len(pk_bytes) != PK_SIZE:
            raise ValueError(f"pk must be {PK_SIZE} bytes")
        if len(m_bytes) != M_SIZE:
            raise ValueError(f"m must be {M_SIZE} bytes")
        self._ensure_idle()
        self.pk[:] = np.frombuffer(pk_bytes, dtype=np.uint8)
        self.m[:] = np.frombuffer(m_bytes, dtype=np.uint8)
        self.pk.flush()
        self.m.flush()
        self.ip.write(REG_CTRL, CTRL_START_ENCAPS)
        cycles = self._wait_done(
            timeout_s if timeout_s is not None else self.DEFAULT_TIMEOUTS_S["encaps"]
        )
        self.ct.invalidate()
        self.ss.invalidate()
        return bytes(self.ct), bytes(self.ss), cycles

    def decaps(self, sk_bytes, ct_bytes, timeout_s=None):
        if len(sk_bytes) != SK_SIZE:
            raise ValueError(f"sk must be {SK_SIZE} bytes")
        if len(ct_bytes) != CT_SIZE:
            raise ValueError(f"ct must be {CT_SIZE} bytes")
        self._ensure_idle()
        self.sk[:] = np.frombuffer(sk_bytes, dtype=np.uint8)
        self.ct[:] = np.frombuffer(ct_bytes, dtype=np.uint8)
        self.sk.flush()
        self.ct.flush()
        self.ip.write(REG_CTRL, CTRL_START_DECAPS)
        cycles = self._wait_done(
            timeout_s if timeout_s is not None else self.DEFAULT_TIMEOUTS_S["decaps"]
        )
        self.ss.invalidate()
        return bytes(self.ss), cycles

    def close(self):
        for buf in (self.pk, self.sk, self.ct, self.ss, self.m):
            try:
                buf.freebuffer()
            except Exception:
                pass

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


In [ ]:
def cycles_to_us(cycles, clk_hz=PL_CLK_HZ):
    return cycles / clk_hz * 1e6


In [ ]:
import os

bitfile = os.environ.get("ML_KEM_BIT", "./ml_kem.bit")
ip_name = os.environ.get("ML_KEM_IP_NAME", "ml_kem_top_0")

print(f"bitfile: {bitfile}")
print(f"ip_name: {ip_name}")


In [ ]:
with MLKem768(bitfile, ip_name=ip_name) as kem:
    status = kem.ip.read(REG_STATUS)
    print(f"STATUS @ reset = 0x{status:x}")
    assert status & STATUS_IDLE, "IP is not idle after overlay load"

    d = bytes(range(32))
    z = bytes(range(32, 64))
    m = bytes(range(0x80, 0xA0))

    pk, sk, c1 = kem.keygen(d, z)
    print(f"KeyGen cycles: {c1} ({cycles_to_us(c1):.1f} us)")

    ct, ss_enc, c2 = kem.encaps(pk, m)
    print(f"Encaps cycles: {c2} ({cycles_to_us(c2):.1f} us)")

    ss_dec, c3 = kem.decaps(sk, ct)
    print(f"Decaps cycles: {c3} ({cycles_to_us(c3):.1f} us)")

    assert ss_enc == ss_dec, "Round-trip ss mismatch"
    print("Smoke test PASSED")
    print(f"Total cycles: {c1 + c2 + c3}")
